# Task: 记忆组织

## Objective / 目标
- 完成一个可用的记忆组织模块

## Scope & Constraints / 范围与约束
- 中文：一个 Notebook 只做一个任务闭环（输入 → 方法 → 输出/验证）。
- English: One notebook = one task loop (input → method → output/validation).
- 核心逻辑应可迁移为 `.py`：尽量纯函数、无隐式全局状态，I/O 放在边缘。

## I/O Spec / 输入输出约定（可按需删改）

| Name | Type | Description |
|------|------|-------------|
| input  | Any | 输入（路径/对象/数组/表等） / Input data |
| result | Any | 输出（对象/指标/产物） / Result / metrics / artifact |

In [ ]:
import sys
import platform
import random
import numpy as np
from datetime import datetime, timedelta
import json
import uuid
from typing import List, Dict, Any, Optional, Tuple, Set
from dataclasses import dataclass, field, asdict
from enum import Enum
import hashlib
from collections import defaultdict

In [ ]:
class MemoryType(Enum):
    """记忆类型枚举"""
    USER_MEMORY = "UserMemory"
    SYSTEM_MEMORY = "SystemMemory"
    FACT = "fact"
    EVENT = "event"
    PREFERENCE = "preference"


class MemoryStatus(Enum):
    """记忆状态枚举"""
    ACTIVATED = "activated"
    ARCHIVED = "archived"
    DEPRECATED = "deprecated"


class RelationType(Enum):
    """关系类型枚举"""
    CAUSES = "causes"           # 因果关系
    FOLLOWS = "follows"         # 时间顺序
    RESOLVES = "resolves"       # 解决关系
    CONTAINS = "contains"       # 包含关系
    RELATED_TO = "related_to"   # 语义相关
    SAME_TOPIC = "same_topic"   # 同主题
    DEPENDS_ON = "depends_on"   # 依赖关系


@dataclass
class MemoryNode:
    """
    记忆节点：系统的基本存储单元
    
    Attributes:
        id: 唯一标识符
        key: 记忆关键词/标题
        memory: 记忆内容摘要
        tags: 标签列表
        type: 记忆类型
        confidence: 置信度 (0-1)
        created_at: 创建时间
        updated_at: 更新时间
        user_id: 用户ID
        session_id: 会话ID
        background: 背景信息
        source_type: 来源类型
        embedding: 向量表示（可选）
    """
    id: str
    key: str
    memory: str
    tags: List[str]
    type: str
    confidence: float
    created_at: str
    updated_at: str
    user_id: str
    session_id: str
    background: str = ""
    source_type: str = "unknown"
    embedding: Optional[List[float]] = None
    
    def to_dict(self) -> Dict:
        """转换为字典"""
        return asdict(self)
    
    @classmethod
    def from_dict(cls, data: Dict) -> 'MemoryNode':
        """从字典创建"""
        return cls(**{k: v for k, v in data.items() if k in cls.__dataclass_fields__})
    
    def get_timestamp(self) -> datetime:
        """获取创建时间的datetime对象"""
        return datetime.fromisoformat(self.created_at.replace('Z', '+00:00'))


@dataclass
class EventUnit:
    """
    事件单元：结构化的事件表示
    
    Attributes:
        event_id: 事件唯一标识
        agent: 执行主体
        action: 动作
        object: 作用对象
        outcome: 结果
        context: 上下文
        timestamp: 时间戳
        source_memory_id: 来源记忆ID
        confidence: 置信度
    """
    event_id: str
    agent: Optional[str]
    action: str
    object: Optional[str]
    outcome: Optional[str]
    context: Optional[str]
    timestamp: str
    source_memory_id: str
    confidence: float = 0.9


@dataclass
class Edge:
    """
    边：连接两个节点的关系
    
    Attributes:
        source_id: 源节点ID
        target_id: 目标节点ID
        relation_type: 关系类型
        weight: 权重/置信度
        metadata: 额外元数据
    """
    source_id: str
    target_id: str
    relation_type: RelationType
    weight: float = 1.0
    metadata: Dict = field(default_factory=dict)


@dataclass
class Session:
    """
    会话/片段：时间连续的记忆集合
    
    Attributes:
        session_id: 会话ID
        memory_ids: 包含的记忆ID列表
        start_time: 开始时间
        end_time: 结束时间
        topic: 会话主题
    """
    session_id: str
    memory_ids: List[str]
    start_time: str
    end_time: str
    topic: Optional[str] = None


@dataclass
class Phase:
    """
    阶段：更高层次的时间划分
    
    Attributes:
        phase_id: 阶段ID
        label: 阶段标签
        session_ids: 包含的会话ID列表
        start_time: 开始时间
        end_time: 结束时间
        summary: 阶段摘要
    """
    phase_id: str
    label: str
    session_ids: List[str]
    start_time: str
    end_time: str
    summary: Optional[str] = None


@dataclass
class AbstractionNode:
    """
    抽象节点：从具体案例中提炼的模式
    
    Attributes:
        node_id: 节点ID
        label: 模式名称
        condition: 适用条件
        solution: 建议方案
        verification: 验证方法
        support_ids: 支撑案例ID列表
        confidence: 置信度
    """
    node_id: str
    label: str
    condition: str
    solution: str
    verification: str
    support_ids: List[str]
    confidence: float = 0.8


# %%
# 配置类 / Configuration
@dataclass
class MemoryOrganizationConfig:
    """记忆组织配置"""
    # 时间结构参数
    time_threshold_minutes: int = 30      # 会话切分的时间阈值
    topic_drift_threshold: float = 0.3    # 主题漂移阈值（降低以减少过度切分）
    context_window_hours: int = 2         # 上下文窗口大小
    
    # 事件结构参数
    event_confidence_threshold: float = 0.5  # 降低事件置信度阈值
    
    # 关系建模参数
    similarity_threshold: float = 0.3     # 降低相似度阈值
    edge_confidence_threshold: float = 0.5  # 降低边置信度阈值
    max_candidates: int = 50
    
    # 层级抽象参数
    min_cluster_size: int = 2             # 最小聚类大小
    abstraction_threshold: float = 0.4    # 降低抽象阈值
    
    # 调试参数
    verbose: bool = False
    use_mock_llm: bool = True

cfg = MemoryOrganizationConfig(verbose=True, use_mock_llm=True)

In [ ]:

def extract_event(text: str, memory_id: str, timestamp: str):
    pass

def infer_relation(self, event_a:Any, event_b: Any):
    """
    推断事件之间的关系
    """
    return RelationType.CAUSES

In [ ]:
class MemoryOrganizer:
    """
    记忆组织器：整合所有模块的主入口
    """
